#  ⭐ `dim_dose_frequencia`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root,  normalize_dose_freq

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet") 
bronze =  bronze[["FREQUENCIA_DOSE"]].drop_duplicates()
bronze.columns

Index(['FREQUENCIA_DOSE'], dtype='object')

In [4]:
bronze.FREQUENCIA_DOSE.nunique()

1638

In [5]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_DOSE_FREQUENCIA.csv", sep=";", index=False)

In [6]:
bronze.head(40)

,FREQUENCIA_DOSE
0,6 hora
1,None
10,cíclico
12,4 hora
14,1 cíclico
16,1 dia
27,1
43,
65,12 hora
71,1 por 1 dia


# 🥈 Silver

normalized data, medium quality


In [7]:
silver = normalize_dose_freq(bronze, col="FREQUENCIA_DOSE")
silver.head()

,FREQUENCIA_DOSE,FREQUENCIA_DOSE_DOSES,FREQUENCIA_DOSE_INTERVALO_VALOR,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
0,6 hora,1.0,6.0,3,hora
1,None,NaN,NaN,0,desconhecido
10,cíclico,1.0,NaN,9,ciclico
12,4 hora,1.0,4.0,3,hora
14,1 cíclico,1.0,NaN,9,ciclico


In [8]:
silver[['FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE', 'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR']].drop_duplicates().sort_values(by='FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR')

,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
19443,7,ano
10,9,ciclico
281969,8,decada
1,0,desconhecido
16,4,dia
0,3,hora
566,6,mes
4432,2,minuto
4545,1,segundo
1037,5,semana


In [9]:
silver.FREQUENCIA_DOSE_INTERVALO_VALOR.value_counts(dropna=False).head(10)

FREQUENCIA_DOSE_INTERVALO_VALOR
1.0     232
NaN     118
2.0     117
8.0     115
6.0     104
3.0     100
4.0      98
12.0     81
21.0     52
15.0     47
Name: count, dtype: int64

In [10]:
silver.FREQUENCIA_DOSE_DOSES.value_counts(dropna=False).head(10)

FREQUENCIA_DOSE_DOSES
1.0     552
2.0     127
3.0      94
4.0      87
6.0      63
5.0      52
8.0      39
12.0     38
7.0      32
10.0     29
Name: count, dtype: int64

In [11]:
silver.query("FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE == 'desconhecido'").FREQUENCIA_DOSE.value_counts(dropna=False).head(40)

Series([], Name: count, dtype: int64)

In [12]:
hist = silver

In [13]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_dose_frequencia/hist_dim_dose_frequencia.parquet", index=False)

In [14]:
hist.to_csv(Path(project_root) / "data/02_silver/hist_dim_dose_frequencia/hist_dim_dose_frequencia_MANUAL.csv", sep=";",index=False)

# 🥇 Gold


In [15]:
hist.columns

Index(['FREQUENCIA_DOSE', 'FREQUENCIA_DOSE_DOSES',
       'FREQUENCIA_DOSE_INTERVALO_VALOR',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR'],
      dtype='object')

In [16]:
dim = hist[['FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR']].sort_values(by='FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR').drop_duplicates()
dim

,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
233395,7,ano
234870,9,ciclico
495686,8,decada
347464,0,desconhecido
61281,4,dia
390188,3,hora
15139,6,mes
438883,2,minuto
528398,1,segundo
37319,5,semana


In [17]:
dim.to_parquet(Path(project_root) / "data/03_gold/dim_dose_frequencia/dim_dose_frequencia.parquet", index=False)